# XGBoost Classifier as a baseline

In [12]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
                            classification_report, ConfusionMatrixDisplay)



## <font color="orange"> Data Loading <font>

In [13]:
path = "../data/extraction/meta_mean_std_mfcc.csv"
df = pd.read_csv(path)
df.head()

,file,duration,transcription,speaker,gender,age,emotion,emotion.confidence,emotion.naturalness,mfcc_mean_1,...,delta_std_11,delta_std_12,delta_std_13,delta_std_14,delta_std_15,delta_std_16,delta_std_17,delta_std_18,delta_std_19,delta_std_20
0,/home/anh-dung-le/audb/emodb/2.0.0/d3b62a9b/wa...,0 days 00:00:01.898250000,a01,3,male,31,happiness,0.90,0.75,-321.306000,...,1.550109,1.159719,1.152341,1.352440,1.158564,0.969121,0.849382,0.844728,0.807716,0.814517
1,/home/anh-dung-le/audb/emodb/2.0.0/d3b62a9b/wa...,0 days 00:00:01.611250000,a01,3,male,31,neutral,1.00,0.95,-322.462976,...,1.518675,1.496530,1.123413,0.970171,1.484902,0.694235,1.298124,0.883971,1.067266,0.582751
2,/home/anh-dung-le/audb/emodb/2.0.0/d3b62a9b/wa...,0 days 00:00:01.877812500,a01,3,male,31,anger,0.95,0.90,-300.835980,...,1.580734,1.366172,1.259567,1.291065,1.324476,1.249770,1.202631,0.821312,0.877034,0.835950
3,/home/anh-dung-le/audb/emodb/2.0.0/d3b62a9b/wa...,0 days 00:00:02.006250000,a02,3,male,31,happiness,0.85,0.60,-305.514536,...,1.392360,1.240131,1.051630,1.184306,1.127038,1.075541,1.038948,0.906845,1.230960,0.893336
4,/home/anh-dung-le/audb/emodb/2.0.0/d3b62a9b/wa...,0 days 00:00:01.439812500,a02,3,male,31,neutral,1.00,0.95,-305.962147,...,1.247700,1.187863,1.010545,0.838740,1.183253,0.725776,0.922650,0.753378,0.695016,0.617318


## <font color="orange"> Feature Selection and Split <font>

In [14]:
# keep mfcc-related cols
stat_cols = [c for c in df.columns if c.startswith(("mfcc_mean_", "mfcc_std_", "delta_mean_", "delta_std_"))]
X = df[stat_cols].copy()

# metadata cols also
X["gender"] = (df["gender"] == "male").astype(int)
X["age"] = df["age"].astype(float)

In [15]:
# show first few rows
X.head()

,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,mfcc_mean_4,mfcc_mean_5,mfcc_mean_6,mfcc_mean_7,mfcc_mean_8,mfcc_mean_9,mfcc_mean_10,...,delta_std_13,delta_std_14,delta_std_15,delta_std_16,delta_std_17,delta_std_18,delta_std_19,delta_std_20,gender,age
0,-321.306000,9.669931,-41.315235,33.774169,-29.929212,19.328474,-23.542137,4.895682,-9.851787,1.101428,...,1.152341,1.352440,1.158564,0.969121,0.849382,0.844728,0.807716,0.814517,1,31.0
1,-322.462976,30.320147,-42.097983,31.981602,-25.557349,17.140117,-19.730400,7.482372,-11.211044,3.936103,...,1.123413,0.970171,1.484902,0.694235,1.298124,0.883971,1.067266,0.582751,1,31.0
2,-300.835980,0.977344,-56.865403,25.825857,-30.137908,15.278820,-21.376164,4.381688,-14.007864,0.172622,...,1.259567,1.291065,1.324476,1.249770,1.202631,0.821312,0.877034,0.835950,1,31.0
3,-305.514536,-0.129116,-44.507566,29.120147,-33.680070,19.283297,-26.528275,5.708885,-16.288680,-1.401260,...,1.051630,1.184306,1.127038,1.075541,1.038948,0.906845,1.230960,0.893336,1,31.0
4,-305.962147,26.578828,-36.236041,37.371035,-22.652406,16.741552,-20.569939,6.261011,-12.960843,1.979905,...,1.010545,0.838740,1.183253,0.725776,0.922650,0.753378,0.695016,0.617318,1,31.0


In [16]:
# target: emotion labels
# encoding
le = LabelEncoder()
y = le.fit_transform(df["emotion"])          # XGBoost needs integer labels


In [17]:
# get sample weights from confidence
w = df["emotion.confidence"].astype(float)
w = (w / w.mean()).values

In [18]:
# consider 4 configurations
configs = {
    "acoustic only":  stat_cols,                     # only mfcc features
    "+ gender":       stat_cols + ["gender"],        # add gender
    "+ age":          stat_cols + ["age"],           # add age
    "+ gender + age": stat_cols + ["gender", "age"], # add both
}

In [19]:
# split: leave one speaker out 
groups = df["speaker"].values
logo = LeaveOneGroupOut()


## <font color="orange"> Modelling: no tune <font>

In [25]:
# definition
clf = XGBClassifier(
    objective="multi:softprob",
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.6,
    colsample_bytree=0.55,
    reg_lambda=2.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

In [ ]:
# train for each config
for name, cols in configs.items():
    acc = []
    pre = []
    rec = []
    f1 = []
    acc_tr = []
    pre_tr = []
    rec_tr = []
    f1_tr = []
    for tr, te in logo.split(X, y, groups):
        m = XGBClassifier(**clf.get_params())
        cw = compute_sample_weight("balanced", y[tr])
        w_comb = w[tr] * cw
        w_comb = w_comb / w_comb.mean()
        m.fit(X.iloc[tr][cols], y[tr], sample_weight=w_comb)
        y_pred = m.predict(X.iloc[te][cols])
        y_pred_train = m.predict(X.iloc[tr][cols])
        acc.append(accuracy_score(y[te], y_pred))
        pre.append(precision_score(y[te], y_pred, average="macro", zero_division=0.0))
        rec.append(recall_score(y[te], y_pred, average="macro", zero_division=0.0))
        f1.append(f1_score(y[te], y_pred, average="macro", zero_division=0.0))
        acc_tr.append(accuracy_score(y[tr], y_pred_train))
        pre_tr.append(precision_score(y[tr], y_pred_train, average="macro", zero_division=0.0))
        rec_tr.append(recall_score(y[tr], y_pred_train, average="macro", zero_division=0.0))
        f1_tr.append(f1_score(y[tr], y_pred_train, average="macro", zero_division=0.0))
    

In [ ]:
for name, cols in configs.items():
    print(f"{name:18s}")
    print("  Test")
    print(f"\t accuracy: {np.mean(acc):.3f} +/- {np.std(acc):.3f}")
    print(f"\t precision (macro): {np.mean(pre):.3f} +/- {np.std(pre):.3f}")
    print(f"\t recall (macro):{np.mean(rec):.3f} +/- {np.std(rec):.3f}")
    print(f"\t f1 (macro): {np.mean(f1):.3f} +/- {np.std(f1):.3f}")
    print("  Train")
    print(f"\t accuracy: {np.mean(acc_tr):.3f} +/- {np.std(acc_tr):.3f}")
    print(f"\t precision (macro): {np.mean(pre_tr):.3f} +/- {np.std(pre_tr):.3f}")
    print(f"\t recall (macro):{np.mean(rec_tr):.3f} +/- {np.std(rec_tr):.3f}")
    print(f"\t f1 (macro): {np.mean(f1_tr):.3f} +/- {np.std(f1_tr):.3f}")

acoustic only     
  Test
	 accuracy: 0.649 +/- 0.083
	 precision (macro): 0.654 +/- 0.102
	 recall (macro):0.590 +/- 0.114
	 f1 (macro): 0.575 +/- 0.104
  Train
	 accuracy: 1.000 +/- 0.000
	 precision (macro): 1.000 +/- 0.000
	 recall (macro):1.000 +/- 0.000
	 f1 (macro): 1.000 +/- 0.000
+ gender          
  Test
	 accuracy: 0.649 +/- 0.083
	 precision (macro): 0.654 +/- 0.102
	 recall (macro):0.590 +/- 0.114
	 f1 (macro): 0.575 +/- 0.104
  Train
	 accuracy: 1.000 +/- 0.000
	 precision (macro): 1.000 +/- 0.000
	 recall (macro):1.000 +/- 0.000
	 f1 (macro): 1.000 +/- 0.000
+ age             
  Test
	 accuracy: 0.649 +/- 0.083
	 precision (macro): 0.654 +/- 0.102
	 recall (macro):0.590 +/- 0.114
	 f1 (macro): 0.575 +/- 0.104
  Train
	 accuracy: 1.000 +/- 0.000
	 precision (macro): 1.000 +/- 0.000
	 recall (macro):1.000 +/- 0.000
	 f1 (macro): 1.000 +/- 0.000
+ gender + age    
  Test
	 accuracy: 0.649 +/- 0.083
	 precision (macro): 0.654 +/- 0.102
	 recall (macro):0.590 +/- 0.114
	 f1 (